In [ ]:
from src.pipelines.lunarlander.visualizers import visualize_dataset
from src.conf.environment import HopperConfig, Walker2dConfig
import minari
config = HopperConfig()
# config = Walker2dConfig()

minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
minari_dataset_stats = get_dataset_stats(minari_dataset)
visualize_dataset(minari_dataset)
minari_dataset_stats

In [ ]:
# try replaying a random episode
from src.pipelines.hopper.visualizers import visualize_trajectory as hvt

minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
minari_dataset_stats = get_dataset_stats(minari_dataset)
long_episode = random.choice(minari_dataset.filter_episodes(lambda x: len(x.actions) > 950))
expert_obs = long_episode.observations
hvt(expert_obs, horizon=100)

In [ ]:
@dataclass
class HopperArgs:
    environment: str = "Hopper-v4"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 128
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs"
    model_type: str = "ccnn"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
args = HopperArgs()

# Loading dataset
minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = (obs_dim + action_dim) * args.horizon
cond_dim = obs_dim
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_start_obs"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

In [ ]:
# Try training with refactored code
# logger = WandBLogger(
#     config=args,
#     run_name=run_name
# )
# model, stats, input_dim = train(
#     config=config, args=args, dataset=minari_dataset, logger=logger
# )
# logger.finish()

In [ ]:
env = minari_dataset.recover_environment()
print(f"Loading model from {MODEL_SAVE_PATH}...")
state_dict = torch.load(MODEL_SAVE_PATH, map_location=device)
vf = ConditionalCNN(
    horizon=args.horizon,
    input_dim=input_dim,
    hidden_dim=args.hidden_dim,
    cond_dim=cond_dim,
    kernel_size=args.kernel_size,
).to(device)
vf.load_state_dict(state_dict)
wrapped_vf = WrappedConditionalModel(vf)
step_size = 0.05
T = torch.linspace(0, 1, 10)  # sample times
T = T.to(device=device)
solver = ODESolver(velocity_model=wrapped_vf)  # create an ODESolver class

env = minari_dataset.recover_environment()
start_observation, _ = env.reset()
start_obs_tensor = torch.from_numpy(start_observation)
obs, act = generate_trajectory(
    stats = minari_dataset_stats,
    solver = solver,
    T = T, 
    input_dim = input_dim,
    args = args,
    horizon = args.horizon,
    condition={args.condition_on: start_obs_tensor},
    batch_size = args.inference_batch_size
)
hvt(obs, horizon=args.horizon, verbose=False)
hvt(expert_obs, horizon=args.horizon, verbose=False)

In [ ]:
# MPC evaluation
env = minari_dataset.recover_environment(eval_env=True, render_mode="human")
num_eval_episodes = 10
mpc_planner = lambda obs: generate_trajectory(
    stats=minari_dataset_stats,
    solver=solver,
    T=T,
    input_dim=input_dim,
    args=args,
    horizon=args.horizon,
    condition={args.condition_on: obs},
    batch_size=args.inference_batch_size,
)
model_rewards = evaluate_policy_mpc(
    env, mpc_planner, num_eval_episodes, replan_freq=1, render=False, max_episode_length=1000
)
env.close()
avg_model_reward = np.mean(model_rewards)
std_model_reward = np.std(model_rewards)
print(
    f"Average MPC Reward over {num_eval_episodes} episodes: {avg_model_reward:.2f} +/- {std_model_reward:.2f}"
)